# Seq2Seq Training — Drilling Operations (E2E pipeline)

Loads preprocessed data from `Data/Data for model (E2E)/{strategy}/`, builds sequences with a **single target shift** (in `training/data.py`), trains the encoder-decoder model with scheduled sampling, evaluates with both top-1 and top-3 accuracy, and saves the model bundle that `inference/` consumes.

**Critical invariants:**
- Target shift happens exactly once (in `build_seq2seq_sequences`).
- Well boundaries are padded with the `End of Operations` sentinel (class index is stable thanks to fixed LabelEncoder fit in preprocessing).
- Pipeline config lives in `config/pipeline.yaml`.

In [ ]:
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Colab bootstrap: mount Drive, git pull, symlink Data/models/results,
    # chdir to repo root, confirm GPU is visible.
    from google.colab import drive, userdata
    import subprocess

    REPO_OWNER = "rovshanhasanov-commits"
    REPO_NAME  = "drilling-predictions"                        # GitHub repo name
    DRIVE_ROOT = "/content/drive/MyDrive/drilling-e2e"         # Drive folder name

    drive.mount('/content/drive')

    PAT = userdata.get('GITHUB_PAT')
    repo_url = f"https://{PAT}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.exists('/content/repo'):
        r = subprocess.run(['git', 'clone', repo_url, '/content/repo'],
                           capture_output=True, text=True)
        if r.returncode != 0:
            safe = r.stderr.replace(PAT, '<REDACTED_PAT>')
            print(f"✗ git clone failed:\n{safe}")
            raise SystemExit(1)
        print("✓ Cloned repo")
    else:
        subprocess.run(['git', '-C', '/content/repo', 'pull'], check=True)
        print("✓ Pulled latest")

    PROJECT = "/content/repo"

    # Data symlink OUTSIDE the repo — pipeline.yaml uses `../Data/` relative to REPO_ROOT
    local_data = "/content/Data"
    drive_data = f"{DRIVE_ROOT}/Data"
    if os.path.islink(local_data) and os.readlink(local_data) != drive_data:
        os.unlink(local_data)
    if not os.path.exists(local_data):
        os.symlink(drive_data, local_data)
        print(f"✓ Data symlink: {local_data} -> {drive_data}")

    # models/ and results/ INSIDE the repo so runs persist to Drive across sessions
    for name in ("models", "results"):
        drive_path = f"{DRIVE_ROOT}/{name}"
        os.makedirs(drive_path, exist_ok=True)
        local_path = f"{PROJECT}/{name}"
        if os.path.exists(local_path) and not os.path.islink(local_path):
            import shutil; shutil.rmtree(local_path)
        if not os.path.islink(local_path):
            os.symlink(drive_path, local_path)
            print(f"✓ {name} symlink: {local_path} -> {drive_path}")

    os.chdir(PROJECT)
    sys.path.insert(0, PROJECT)
    print(f"✓ cwd: {os.getcwd()}")

else:
    # Local (Windows / macOS / Linux): just make sure cwd is the repo root
    REPO_ROOT = Path.cwd().resolve()
    if REPO_ROOT.name != 'Drilling - End to End Prediction':
        REPO_ROOT = REPO_ROOT.parent
    os.chdir(REPO_ROOT)
    sys.path.insert(0, str(REPO_ROOT))
    print(f"local — repo root: {REPO_ROOT}")

# GPU sanity (harmless on CPU)
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    print(f"TensorFlow: {len(gpus)} GPU(s) visible — {gpus or '(CPU mode)'}")
except Exception as e:
    print(f"TF GPU check skipped: {e}")

In [ ]:
# Smoke test — verify config loads and the target parquet is reachable.
# Fails fast here if the Drive mount or symlink is off, rather than deep in the training loop.
import pickle
import pandas as pd
from config import load_config, resolve

cfg = load_config()
_strategy = cfg['training']['embedding_strategy']
_data_dir = resolve(cfg, cfg['data']['output_dir']) / _strategy

_train_path = _data_dir / 'df_train.parquet'
assert _train_path.exists(), f"MISSING: {_train_path} — check Data upload / symlink"
_df = pd.read_parquet(_train_path)
print(f"✓ df_train ({_strategy}): {len(_df):,} rows × {_df.shape[1]} cols")

_label_flags = [c for c in _df.columns if c.endswith('_label_real')]
print(f"✓ label_real columns: {_label_flags}")

for split in ('val', 'test'):
    _p = _data_dir / f'df_{split}.parquet'
    assert _p.exists(), f"MISSING: {_p}"
    print(f"✓ df_{split}.parquet present")

with open(_data_dir / 'encoders.pkl', 'rb') as f:
    _enc = pickle.load(f)
_per_head = {h: len(_enc['target_encoders'][h].classes_)
             for h in ['phase', 'phase_step', 'major_ops_code', 'operation']}
print(f"✓ encoders.pkl — class counts per head: {_per_head}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from config import load_config, resolve
from training.data import (
    load_strategy_data, compute_numeric_cols, build_seq2seq_sequences,
    eoo_encoded_ids, eoo_duration_value, HIERARCHY,
)
from training.model import build_seq2seq_model
from training.train import prep_encoder_inputs, prep_model_inputs, prep_targets, train
from training.evaluate import per_step_accuracy, autoregressive_predict
from training.save_artifacts import save_bundle

cfg = load_config()
tcfg = cfg['training']
print(f"pipeline.yaml loaded -- baseline cfg['training'] has {len(tcfg)} keys")

In [ ]:
# ============================================================
# Runtime config overrides — paste AFTER `cfg = load_config()`,
# BEFORE the model-building cell. Each line left commented = use
# the pipeline.yaml default. Uncomment + edit what you want to sweep.
# ============================================================
tcfg = cfg['training']

# ---- Learning rate & schedule ----------------------------------------------
# tcfg['learning_rate']            = 1e-5     # default: 0.0001 (initial LR; lr_max for cosine)
# tcfg['lr_schedule']              = 'plateau'        # default: 'plateau' | 'cosine_restarts'

# Plateau-mode knobs (used when lr_schedule == 'plateau'):
tcfg['lr_patience']              = 3        # default: 5    (epochs of no val_loss improvement before LR halves; 0 disables)
# tcfg['lr_factor']                = 0.5      # default: 0.5  (multiplier per drop)
# tcfg['min_lr']                   = 1e-7     # default: 1e-7 (floor)

# Cosine warm restarts knobs (used when lr_schedule == 'cosine_restarts'):
# tcfg['cosine_t_0']               = 50       # default: 50   (first cycle length, in epochs)
# tcfg['cosine_t_mult']            = 2        # default: 2    (cycle length multiplier per restart)
# tcfg['cosine_min_lr']            = 1e-7     # default: 1e-7

# ---- Early stopping & epochs -----------------------------------------------
# tcfg['epochs']                   = 500      # default: 500
tcfg['early_stopping_patience']  = 20       # default: 30   (epochs of no val_loss improvement before stopping)

# ---- Batch size & sequence shape -------------------------------------------
# tcfg['batch_size']               = 64       # default: 64   (T4 can probably handle 128 or 256 for embed_separate/state)
tcfg['sequence_length']          = 15       # default: 25   (encoder context window)
# tcfg['n_future']                 = 8        # default: 8    (prediction horizon; 15 ≈ 24h of ops)

# ---- Embedding strategy & target heads -------------------------------------
# tcfg['embedding_strategy']       = 'dummies'        # default: 'dummies'; options: 'dummies' | 'embed_separate' | 'embed_state'
# tcfg['target_variables']         = [                # default: hierarchy 4 + duration_bin_next
#     'phase_next', 'phase_step_next', 'major_ops_code_next', 'operation_next',
#     'duration_next',                                # uncomment to use the regression duration head
#     'duration_bin_next',                            # uncomment to use the classification (bin) duration head
# ]
# Note: keep exactly one of `duration_next` / `duration_bin_next` active per run
# (the other commented). Both can technically coexist (warn-but-allow), but the
# regression head wins for StepPrediction.duration_hours when both are on.

# ---- Scheduled sampling ----------------------------------------------------
# tcfg['ss_start_rate']            = 0.1      # default: 0.0
# tcfg['ss_end_rate']              = 0.75     # default: 0.75
# tcfg['ss_ramp_epochs']           = 200      # default: 200  (epochs to ramp ss_start → ss_end)

# ---- Architecture ----------------------------------------------------------
tcfg['enc_lstm_units']           = [256, 128, 64]        # default: [128, 64]  (stacked encoder widths)
# tcfg['dec_lstm_units']           = 128              # default: 128        (decoder LSTM width)
# tcfg['dense_units']              = [64]             # default: [64]       (head projection layers)
# tcfg['dropout_rate']             = 0.3              # default: 0.3

# ---- Decoder target embedding sizes ----------------------------------------
# tcfg['dec_target_edims']['phase_next']           = 4     # default: 4
# tcfg['dec_target_edims']['phase_step_next']      = 8     # default: 8
# tcfg['dec_target_edims']['major_ops_code_next']  = 16    # default: 16
# tcfg['dec_target_edims']['operation_next']       = 32    # default: 32
# tcfg['dec_target_edims']['duration_bin_next']    = 8     # default: 8 (10 classes: 7 bins + EOO + Unplanned + UNK)

# ---- Loss head weights (relative importance across heads) ------------------
tcfg['loss_weights']['phase_next']          = 0.2   # default: 0.2
tcfg['loss_weights']['phase_step_next']     = 0.3   # default: 0.3
tcfg['loss_weights']['major_ops_code_next'] = 1.0   # default: 1.0
tcfg['loss_weights']['operation_next']      = 1.5   # default: 2.0
tcfg['loss_weights']['duration_next']       = 1.0   # default: 1.5 (only used if duration_next in target_variables)
# tcfg['loss_weights']['duration_bin_next']   = 1.0   # default: 1.0 (CE loss; only used if duration_bin_next in target_variables)

# ---- Recompute derived constants so downstream cells see overrides ---------
STRATEGY = tcfg['embedding_strategy']
SEQ_LEN  = tcfg['sequence_length']
N_FUTURE = tcfg['n_future']
TOP_K    = tcfg['top_k_for_eval']
ACTIVE   = [t.replace('_next', '') for t in tcfg['target_variables'] if t != 'duration_next']
PREDICT_DURATION = 'duration_next' in tcfg['target_variables']

# ---- Print what's actually in effect for this run --------------------------
from config import model_folder_name
print(f"Run folder pattern: models/{model_folder_name(cfg)}_<TIMESTAMP>/")
print(f"  strategy={STRATEGY}  seq_len={SEQ_LEN}  n_future={N_FUTURE}")
print(f"  targets={ACTIVE}  predict_duration={PREDICT_DURATION}")
sched = tcfg.get('lr_schedule', 'plateau')
if sched == 'cosine_restarts':
    print(f"  lr={tcfg['learning_rate']:g}  patience={tcfg['early_stopping_patience']}  "
          f"lr_schedule=cosineWR T0={tcfg.get('cosine_t_0', 50)} M={tcfg.get('cosine_t_mult', 2)} "
          f"min_lr={tcfg.get('cosine_min_lr', 0):g}")
else:
    print(f"  lr={tcfg['learning_rate']:g}  patience={tcfg['early_stopping_patience']}  "
          f"lr_schedule=plateau halve/{tcfg.get('lr_patience', 0)} epochs to {tcfg.get('min_lr', 0):g}")
print(f"  batch_size={tcfg['batch_size']}  epochs={tcfg['epochs']}")

## 1. Load preprocessed data

In [ ]:
strategy_dir = resolve(cfg, cfg['data']['output_dir']) / STRATEGY
bundle = load_strategy_data(strategy_dir)
df_train, df_val, df_test = bundle['df_train'], bundle['df_val'], bundle['df_test']
encoders = bundle['encoders']
data_config = bundle['config']
cat_input_cols = data_config['cat_input_cols']
numeric_cols = compute_numeric_cols(data_config)
n_classes = encoders['n_classes']
print(f'train/val/test rows: {len(df_train):,}/{len(df_val):,}/{len(df_test):,}')
print(f'cat_input_cols: {cat_input_cols}')
print(f'n_numeric: {len(numeric_cols)}')
print(f'n_classes: {n_classes}')

## 2. Build sequences (single target shift + EOO padding)

In [ ]:
# Targets = every categorical head in ACTIVE — HIERARCHY 4 plus
# 'duration_bin' when the bin classification head is on. Without this,
# prep_targets() KeyErrors trying to read duration_bin_target_enc from y.
target_cols = [f'{t}_target_enc' for t in ACTIVE]
eoo_enc = eoo_encoded_ids(encoders['target_encoders'], encoders['eoo_token'])
eoo_dur = eoo_duration_value(encoders['dur_scaler'])
print('EOO encoded ids:', eoo_enc)
print('EOO duration (scaled):', eoo_dur)

seq_train = build_seq2seq_sequences(df_train, cat_input_cols, numeric_cols, target_cols,
                                     SEQ_LEN, N_FUTURE, eoo_enc, eoo_dur, 'train')
seq_val   = build_seq2seq_sequences(df_val,   cat_input_cols, numeric_cols, target_cols,
                                     SEQ_LEN, N_FUTURE, eoo_enc, eoo_dur, 'val')
seq_test  = build_seq2seq_sequences(df_test,  cat_input_cols, numeric_cols, target_cols,
                                     SEQ_LEN, N_FUTURE, eoo_enc, eoo_dur, 'test')
print(f'sequences train/val/test: {seq_train["num"].shape[0]:,}/{seq_val["num"].shape[0]:,}/{seq_test["num"].shape[0]:,}')
print(f'encoder num shape: {seq_train["num"].shape}')
for k, v in seq_train['y'].items():
    print(f'  target {k}: {v.shape}')

## 3. Build model

In [ ]:
training_model, encoder_model, decoder_step_model = build_seq2seq_model(
    emb_strategy=STRATEGY,
    cat_input_cols=cat_input_cols,
    cat_encoders=encoders['cat_encoders'],
    n_classes=n_classes,
    n_numeric=seq_train['num'].shape[2],
    seq_len=SEQ_LEN,
    n_future=N_FUTURE,
    enc_lstm_units=tcfg['enc_lstm_units'],
    dec_lstm_units=tcfg['dec_lstm_units'],
    dense_units=tcfg['dense_units'],
    dropout=tcfg['dropout_rate'],
    learning_rate=tcfg['learning_rate'],
    loss_weights={k.replace('_next',''): v for k, v in tcfg['loss_weights'].items()},
    active_targets=ACTIVE,
    predict_duration=PREDICT_DURATION,
    dec_target_edims=tcfg['dec_target_edims'],
)
training_model.summary()

## 4. Train with scheduled sampling

In [ ]:
enc_X_train = prep_encoder_inputs(seq_train['cat'], seq_train['num'], cat_input_cols)
enc_X_val   = prep_encoder_inputs(seq_val['cat'],   seq_val['num'],   cat_input_cols)
enc_X_test  = prep_encoder_inputs(seq_test['cat'],  seq_test['num'],  cat_input_cols)

# Loss-masking via sample_weight dicts emitted by build_seq2seq_sequences
# (operation / major_ops_code / duration flags from preprocessing/clean.py).
sw_train = seq_train.get('sample_weight')
sw_val   = seq_val.get('sample_weight')
if sw_train is not None:
    for head, arr in sw_train.items():
        n_masked_train = int((arr == 0).sum())
        n_masked_val   = int((sw_val[head] == 0).sum()) if sw_val else 0
        print(f'  {head:16s} loss mask active: {n_masked_train} train + {n_masked_val} val target slots zeroed')

history_logs, run_metadata = train(
    training_model,
    enc_X_train, seq_train['y'], seq_train['y_dur'],
    enc_X_val,   seq_val['y'],   seq_val['y_dur'],
    active_targets=ACTIVE,
    predict_duration=PREDICT_DURATION,
    n_classes=n_classes,
    batch_size=tcfg['batch_size'],
    epochs=tcfg['epochs'],
    ss_start_rate=tcfg['ss_start_rate'],
    ss_end_rate=tcfg['ss_end_rate'],
    ss_ramp_epochs=tcfg['ss_ramp_epochs'],
    early_stopping_patience=tcfg['early_stopping_patience'],
    sample_weight_train=sw_train,
    sample_weight_val=sw_val,
    lr_schedule=tcfg.get('lr_schedule', 'plateau'),
    lr_patience=tcfg.get('lr_patience', 0),
    lr_factor=tcfg.get('lr_factor', 0.5),
    min_lr=tcfg.get('min_lr', 1e-7),
    cosine_t_0=tcfg.get('cosine_t_0', 50),
    cosine_t_mult=tcfg.get('cosine_t_mult', 2),
    cosine_min_lr=tcfg.get('cosine_min_lr', 1e-7),
)

## 5. Save model bundle for inference

Saves the model + encoders + a rich `model_config.json` (full effective cfg, git SHA, environment versions, dataset fingerprints, run metadata, LR/loss history). Run evaluation separately via `python -m evaluation.run_evaluation --model-dir <path>`.

In [ ]:
from datetime import datetime
from config import get_model_dir

# Timestamp baked into the bundle folder so each training run is its own sibling
# (re-running training never overwrites). Eval / inference will pick up the
# folder by exact path on disk — they don't re-derive the name.
TIMESTAMP = datetime.utcnow().strftime("%Y%m%d_%H%M")
model_dir = get_model_dir(cfg, timestamp=TIMESTAMP)

save_bundle(
    model_dir,
    training_model, encoder_model, decoder_step_model,
    strategy_data_dir=strategy_dir,
    cfg=cfg,
    run_metadata=run_metadata,
    n_classes=n_classes,
    cat_input_cols=cat_input_cols,
    numeric_cols=numeric_cols,
)